# 🖼️ Image Classification: CNN vs. ViT

**CO5085 – Assignment 1**

So sánh: **ResNet-50, EfficientNet-B0** (CNN) vs. **ViT-B/16, DeiT-S** (Vision Transformer)

In [ ]:
import sys
sys.path.insert(0, '../src')
import torch
import os
os.makedirs('../results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Data

In [ ]:
from datasets import get_cifar100_loaders
train_loader, val_loader, test_loader = get_cifar100_loaders(
    data_dir='../data/image', batch_size=128)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 2. Model 1: ResNet-50 (CNN)

In [ ]:
from models import get_resnet50
from train import train

resnet = get_resnet50(num_classes=100, pretrained=True, freeze_backbone=False)
history_resnet = train(resnet, train_loader, val_loader,
                       num_epochs=20, lr=1e-3,
                       device=DEVICE, save_path='../results/resnet50_best.pt',
                       scheduler_type='cosine')

## 3. Model 2: EfficientNet-B0 (CNN)

In [ ]:
from models import get_efficientnet_b0
from train import train

effnet = get_efficientnet_b0(num_classes=100, pretrained=True)
history_effnet = train(effnet, train_loader, val_loader,
                       num_epochs=20, lr=1e-3,
                       device=DEVICE, save_path='../results/efficientnet_best.pt')

## 4. Model 3: ViT-B/16

In [ ]:
from models import get_vit_b16
from datasets import get_cifar100_loaders
from torchvision import datasets, transforms

# ViT needs 224x224 – use a transform wrapper
train_loader_vit, val_loader_vit, test_loader_vit = get_cifar100_loaders(
    data_dir='../data/image', batch_size=32)  # smaller batch for memory

vit_model = get_vit_b16(num_classes=100, pretrained=True)
history_vit = train(vit_model, train_loader_vit, val_loader_vit,
                    num_epochs=15, lr=2e-5,
                    device=DEVICE, save_path='../results/vit_b16_best.pt',
                    scheduler_type='cosine')

## 5. Model 4: DeiT-Small

In [ ]:
from models import get_deit_small

train_loader_deit, val_loader_deit, test_loader_deit = get_cifar100_loaders(
    data_dir='../data/image', batch_size=32)

deit_model = get_deit_small(num_classes=100, pretrained=True)
history_deit = train(deit_model, train_loader_deit, val_loader_deit,
                     num_epochs=15, lr=2e-5,
                     device=DEVICE, save_path='../results/deit_small_best.pt',
                     scheduler_type='cosine')

## 6. Evaluation & Comparison

In [ ]:
from evaluate import get_predictions, compute_metrics, plot_training_curves, compare_models
import torch

results = {}
for model_name, model, hist, loader in [
    ('ResNet-50',       resnet,      history_resnet, test_loader),
    ('EfficientNet-B0', effnet,      history_effnet, test_loader),
    ('ViT-B/16',        vit_model,   history_vit,    test_loader_vit),
    ('DeiT-Small',      deit_model,  history_deit,   test_loader_deit),
]:
    print(f'\n=== {model_name} ===')
    preds, labels, probs = get_predictions(model, loader, DEVICE)
    results[model_name] = compute_metrics(preds, labels, verbose=True)
    plot_training_curves(hist, model_name=model_name, save_path=f'../results/{model_name}_curves.png')

ModuleNotFoundError: No module named 'evaluate'

## 6. Overall Comparison Chart

In [ ]:
from evaluate import compare_models
compare_models(results, metric='accuracy', save_path='../results/image_comparison_acc.png')
compare_models(results, metric='f1_macro', save_path='../results/image_comparison_f1.png')